In [2]:
from __future__ import annotations

import jax.numpy as jnp
import numpy as np
import dynamiqs as dq

from parameters import DeviceParameters, PulseParameters 
from hamiltonian import readout_hamiltonian, operators
#from operators import operators, jump_operators, basis_state, fluxonium_projector, computational_project

In [ ]:
def qubit_resonator_hamiltonian(params: DeviceParameters) -> dq.QArray:
    # Construct the Hamiltonian for the coupled qubit-resonator system, based on the device parameters. The Hamiltonian includes the resonator term, the fluxonium term, and the dispersive coupling between the two. The Hamiltonian is represented as a QArray, which is a JAX-compatible array type that can be used for efficient numerical simulations.
    ops = operators(params)
    a = ops["a"] # Annihilation operator for the resonator
    n = ops["n"] # Number operator for the fluxonium
    h_resonator = params.resonator_freq * a.dag() @ a # Resonator Hamiltonian term
    h_fluxonium = params.fluxonium_freq * n # Fluxonium Hamiltonian term
    h_coupling = params.dispersive_shift * n @ a.dag() @ a # Dispersive coupling term
    return h_resonator + h_fluxonium + h_coupling

In [6]:
def readout_modulation(pulse: PulseParameters, total_time: float) -> dq.QArray:
    # Compute the time-dependent modulation for the drive term in the Hamiltonian, based on the pulse parameters and the current time t. The modulation is given by the envelope of the pulse multiplied by a cosine function at the drive frequency.
    def modulation(t: float):
        active = jnp.where((t >= 0.0) & (t <= total_time), 1.0, 0.0) # This ensures that the modulation is only active during the time interval of the pulse, and is zero outside of that interval.
        rise = 1.0 if pulse.rise_time <= 0 else jnp.minimum(1.0, t / pulse.rise_time) # Compute the rise envelope of the pulse, which ramps up from 0 to 1 over the specified rise time. If the rise time is zero or negative, we assume an instantaneous rise and set the envelope to 1.
        fall_start = total_time - pulse.fall_time # Compute the start time of the fall envelope, which is the total pulse duration minus the specified fall time.
        fall = (
            1.0
            if pulse.fall_time <= 0
            else jnp.minimum(1.0, jnp.maximum(0.0, (fall_start - t) / pulse.fall_time))
        ) # Compute the fall envelope of the pulse, which ramps down from 1 to 0 over the specified fall time at the end of the pulse duration. If the fall time is zero or negative, we assume an instantaneous fall and set the envelope to 1.
        carrier = jnp.cos(2 * jnp.pi * pulse.drive_freq * t + pulse.phase) # Compute the carrier wave for the drive, which is a cosine function at the specified drive frequency and phase.
        return active * rise * fall * carrier # The overall modulation is given by multiplying these factors together.
    return modulation

In [7]:
def drive_hamiltonian(params: DeviceParameters, pulse: PulseParameters, total_time: float) -> dq.QArray:
    # Construct the time-dependent drive Hamiltonian for the readout pulse, based on the device parameters, pulse parameters, and total simulation time. The drive Hamiltonian includes a modulation function that defines the shape of the pulse envelope, which is active only during the specified time interval of the pulse. The drive term is represented as a QArray, which can be added to the system Hamiltonian for time-dependent simulations.
    ops = operators(params)
    a = ops["a"] # Annihilation operator for the resonator
    modulation = dq.readout_modulation(pulse, total_time) # Get the modulation function for the drive term
    return modulation * (a + a.dag()) # Drive Hamiltonian term, which couples to the resonator field